# Sesión 7 · Series de tiempo y LSTM
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** convertir una serie horaria en ventanas, calcular líneas base y entrenar una LSTM para predecir la siguiente hora.

## 1. Serie horaria de potencia solar

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
horas = pd.date_range('2025-01-01', periods=24*120, freq='h')
h = horas.hour.values
sol = np.clip(np.sin((h-6)/12*np.pi), 0, None)            # ciclo diario del sol
nube = np.clip(rng.normal(0.3, 0.25, len(horas)), 0, 0.9)  # nubosidad
irr = 1000*sol*(1-nube)+rng.normal(0,15,len(horas))
irr = np.clip(irr, 0, None)
temp = 18 + 8*sol + rng.normal(0,1.2,len(horas))
pot = np.clip(0.18*irr*(1-0.004*(temp-25)) + rng.normal(0,2,len(horas)), 0, None)  # kW
df = pd.DataFrame({'irr':irr,'temp':temp,'nube':nube,'hora':h,'pot':pot}, index=horas)
df.head()
plt.figure(figsize=(10,3)); plt.plot(df['pot'].values[:24*5]); plt.show()

## 2. Líneas base
- **Persistencia:** la próxima hora será igual a la actual.
- **Ayer a la misma hora:** el valor de hace 24 horas.
Dividimos por fecha: 80 % train, 20 % test.

In [ ]:
from sklearn.metrics import mean_absolute_error
corte = int(len(df)*0.8)
y = df['pot'].values
# la predicción para la hora i usa solo información anterior
persist = y[corte-1:-1]
ayer = y[corte-24:-24]
real = y[corte:]
print('MAE persistencia   :', round(mean_absolute_error(real, persist),2))
print('MAE ayer misma hora:', round(mean_absolute_error(real, ayer),2))

## 3. Ventanas deslizantes
Cada ejemplo: las 24 horas anteriores de 3 variables (potencia, temperatura, irradiancia). La respuesta: la potencia de la hora siguiente. Forma: `(muestras, 24 pasos, 3 variables)`.

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['pot','temp','irr']
esc = StandardScaler().fit(df[cols].iloc[:corte])
Z = esc.transform(df[cols])
V = 24
X, Y = [], []
for i in range(V, len(Z)):
    X.append(Z[i-V:i]); Y.append(Z[i,0])
X, Y = np.array(X), np.array(Y)
# la muestra i corresponde a la hora i+V; separamos por fecha
lim = corte - V
Xtr, Ytr, Xte, Yte = X[:lim], Y[:lim], X[lim:], Y[lim:]
print('Xtr', Xtr.shape, '| Xte', Xte.shape)

## 4. Modelo clásico de referencia
Una regresión lineal sobre las 72 cifras aplanadas de la ventana. Es el "competidor sencillo" de la LSTM.

In [ ]:
from sklearn.linear_model import Ridge
lin = Ridge(alpha=1.0).fit(Xtr.reshape(len(Xtr),-1), Ytr)
p_lin = lin.predict(Xte.reshape(len(Xte),-1))
def kw(v): return v*esc.scale_[0] + esc.mean_[0]      # volver a kW
print('MAE ridge (kW):', round(mean_absolute_error(kw(Yte), kw(p_lin)),2))

## 5. LSTM con Keras
Una capa LSTM de 32 unidades y una salida. `EarlyStopping` detiene el entrenamiento cuando la validación deja de mejorar. En Colab, TensorFlow ya está instalado.

> Si ve un error de forma, revise que `Xtr` tenga 3 dimensiones.

In [ ]:
p_lstm = None
try:
    import tensorflow as tf
    tf.random.set_seed(0)
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(V, len(cols))),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1)])
    modelo.compile(optimizer='adam', loss='mse')
    parada = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    hist = modelo.fit(Xtr, Ytr, epochs=20, batch_size=64, validation_split=0.2, callbacks=[parada], verbose=0)
    p_lstm = modelo.predict(Xte, verbose=0).ravel()
    print('MAE LSTM (kW):', round(mean_absolute_error(kw(Yte), kw(p_lstm)),2))
    plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val'); plt.legend(); plt.show()
except ImportError:
    print('TensorFlow no está instalado aquí. Use Google Colab para este paso.')

## 6. Tabla comparativa
Llene la tabla con los MAE obtenidos. La LSTM solo se justifica si supera a las líneas base y al modelo lineal.

In [ ]:
res = {'Persistencia': mean_absolute_error(real, persist),
       'Ayer misma hora': mean_absolute_error(real, ayer),
       'Ridge (ventana)': mean_absolute_error(kw(Yte), kw(p_lin))}
if p_lstm is not None:
    res['LSTM'] = mean_absolute_error(kw(Yte), kw(p_lstm))
tabla = pd.Series(res).round(2).sort_values()
print(tabla)
tabla.plot.barh(); plt.xlabel('MAE (kW)'); plt.show()

## 7. Real contra predicho

In [ ]:
n = 24*4
plt.figure(figsize=(10,3.5))
plt.plot(kw(Yte)[:n], label='real')
plt.plot(kw(p_lin)[:n], label='ridge')
if p_lstm is not None: plt.plot(kw(p_lstm)[:n], label='LSTM')
plt.legend(); plt.show()

## 8. Ejercicios
1. Cambie la ventana `V` a 12 y a 48. ¿Cómo cambia el MAE?
2. Quite la irradiancia de `cols`. ¿Qué pasa?
3. Cambie las unidades de la LSTM a 8 y a 128.
4. ¿Qué modelo elegiría y por qué? Considere precisión, simplicidad y tiempo.